In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# import sys
# import asyncio

# # Fix for Windows issues in Jupyter notebooks
# if sys.platform == "win32":
#     # 1. Use ProactorEventLoop for subprocess support
#     if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
#         asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
#     # 2. Redirect stderr to avoid fileno() error when launching MCP servers
#     if "ipykernel" in sys.modules:
#         sys.stderr = sys.__stderr__

<h2>Online MCP server</h2>

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client=MultiServerMCPClient(
     {
        
            "factanker": {
            "transport": "http",
            "url": "https://factanker.com/mcp"
            }
        
}

)
tools = await client.get_tools()

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_groq import ChatGroq
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver



model = ChatGroq(
    model="openai/gpt-oss-20b",
    
)

agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=InMemorySaver()
)

config={'configurable':{"thread_id":"1"}}
Question=HumanMessage(content="What revenue did Exxon file for FY2025?")
response= await agent.ainvoke(
    {"messages":[Question]},
    config
)

from pprint import pprint
pprint(response)

# print(response['messages'][-1].content)

{'messages': [HumanMessage(content='What revenue did Exxon file for FY2025?', additional_kwargs={}, response_metadata={}, id='ab2a1a3e-7635-4b96-a0c4-0a2b35fe3670'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What revenue did Exxon file for FY2025?" We need to provide the revenue figure for Exxon (Exxon Mobil Corp.) for FY2025. That is a future year (FY2025). Exxon hasn\'t filed FY2025 yet; the latest filing would be FY2024. So the answer should state that there\'s no FY2025 filing yet, as it\'s a future year. We must not guess. Use the Factanker tool to verify. We\'ll need to check if there is any filing for FY2025. We can use get_facts or get_timeseries for revenue. Let\'s do get_facts for Exxon Mobil with predicate revenue. That will give the most recent revenue. Then we can check if there is a FY2025. If not, we say no evidence. The user specifically asks "What revenue did Exxon file for FY2025?" We need to answer: No evidence; no F